In [2]:
import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\research


In [3]:
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01")

In [4]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Regression_01'

In [7]:
import box
print(box.__version__)

7.4.1


In [8]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataIngestionConfig:
    root_dir: Path
    dataset_name: str
    local_data_file: Path
    unzip_dir: Path

In [9]:
from Regression_01.constant import *
from Regression_01.utils.common import read_yaml, create_directories

In [10]:
# 4 Update configuration manager

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            dataset_name=config.dataset_name,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [11]:
import os
import zipfile
from pathlib import Path
from Regression_01.logging import logger
from Regression_01.utils.common import get_size

In [12]:
# 5 Components

import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

from Regression_01.logging import logger


class DataIngestion:

    def __init__(self, config):
        self.config = config

    def download_files(self):

        if not os.path.exists(self.config.local_data_file):  #download file from kaggle

            api = KaggleApi()
            api.authenticate() # from kaggle.json

            api.dataset_download_files(
                dataset=self.config.dataset_name,
                path=self.config.root_dir,
                unzip=False # Because Download Extraction are separate responsibilities.
            )

            logger.info("Dataset downloaded successfully")

        else:
            logger.info("Dataset already exists")

    def extract_zip_file(self):

        os.makedirs(self.config.unzip_dir, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

        logger.info("Dataset extracted successfully")

In [13]:
# 6 pipeline
import zipfile
import os
try:
    print("--- STARTING FULL PIPELINE RUN ---")
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()
    
    pipeline_component = DataIngestion(config=data_ingestion_config)
    
    # 1. This will say "File already exists!" since we just downloaded it
    pipeline_component.download_files()
    
    # 2. This will unpack your dataset
    pipeline_component.extract_zip_file()
    
    print("--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---")

except Exception as e:
    raise e

--- STARTING FULL PIPELINE RUN ---
[2026-07-30 01:01:09,873: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\config\config.yaml loaded successfully]
[2026-07-30 01:01:09,885: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Regression_01\params.yaml loaded successfully]
[2026-07-30 01:01:09,889: INFO: common: created directory at artifacts]
[2026-07-30 01:01:09,891: INFO: common: created directory at artifacts/data_ingestion]
Dataset URL: https://www.kaggle.com/datasets/yasserh/housing-prices-dataset
[2026-07-30 01:01:10,970: INFO: 942183107: Dataset downloaded successfully]
[2026-07-30 01:01:10,977: INFO: 942183107: Dataset extracted successfully]
--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---
